# py17: NumPy 字节交换、副本和视图

本 notebook 演示大端/小端模式、字节交换函数、以及副本 (copy) 与视图 (view) 的区别。

In [ ]:
import numpy as np

## 1. 字节序概念

- **大端模式 (Big-Endian)**：高字节存低地址，低字节存高地址（网络传输常用）
- **小端模式 (Little-Endian)**：低字节存低地址，高字节存高地址（x86 架构常用）

例如 int16 的 `0x1234`：
- 大端：`12 34`
- 小端：`34 12`

In [ ]:
# 查看当前系统的字节序
print('系统字节序 =', np.dtype('>i4').byteorder)  # > 表示大端
print('系统本机 =', np.dtype('i4').byteorder)      # = 表示本机

# NumPy 使用的符号：
# >  大端 (Big-Endian)
# <  小端 (Little-Endian)
# =  本机顺序
# |  不适用（单字节类型如 uint8）

## 2. ndarray.byteswap()

将数组中每个元素的字节进行大小端互换。

In [ ]:
# ---- 基本用法 ----
a = np.array([1, 256, 8755], dtype=np.int16)
print('原始数组 =', a)
print('十六进制 =', [hex(x) for x in a])
# [0x1, 0x100, 0x2233]

# byteswap(inplace=True)：原地交换
a.byteswap(inplace=True)
print('交换后 =', a)
print('十六进制 =', [hex(x) for x in a])
# [0x100, 0x1, 0x3322] 高低字节互换

In [ ]:
# ---- byteswap(inplace=False)：返回新数组 ----
a = np.array([1, 256, 8755], dtype=np.int16)
b = a.byteswap(inplace=False)
print('原数组不变 =', a)
print('新数组 =', b)
print('b 十六进制 =', [hex(x) for x in b])

In [ ]:
# ---- 不同数据类型的字节交换 ----
# int32（4字节）
a = np.array([1, 256], dtype=np.int32)
print('int32 原始 =', [hex(x) for x in a])
a.byteswap(inplace=True)
print('int32 交换后 =', [hex(x) for x in a])

# float64（8字节）
a = np.array([1.0, 2.5], dtype=np.float64)
print('float64 原始 =', a)
print('float64 原始 hex =', [hex(x) for x in a.view(np.uint64)])
a.byteswap(inplace=True)
print('float64 交换后 =', a)  # 值会变得很奇怪

## 3. byteswap 的实际用途

字节交换主要用于：
1. 读取不同平台的二进制文件（如大端格式的数据文件）
2. 网络数据传输（网络字节序是大端）
3. 与 C 结构体交互时调整字节对齐

In [ ]:
# ---- 模拟读取大端格式文件 ----
# 假设数据来自大端系统
big_endian_data = np.array([0x0100, 0x0200, 0x0300], dtype='>i2')
print('大端数据 =', big_endian_data)
print('字节序 =', big_endian_data.dtype.byteorder)  # >

# 转换为本机字节序
native_data = big_endian_data.astype(np.int16)
print('本机数据 =', native_data)
print('本机字节序 =', native_data.dtype.byteorder)  # = 或 <

In [ ]:
# ---- 另一种转换方式：newbyteorder ----
a = np.array([1, 2, 3], dtype='>i4')  # 大端
print('原始字节序 =', a.dtype.byteorder)  # >

# 转为小端
b = a.astype(a.dtype.newbyteorder('<'))
print('转换后字节序 =', b.dtype.byteorder)  # <
print('值不变 =', b)

## 4. 字节序与文件 IO

In [ ]:
# ---- 保存和读取时指定字节序 ----
a = np.array([1, 2, 3, 4], dtype=np.int32)
print('本机数据 =', a)

# 保存为大端格式
a.astype('>i4').tofile('big_endian.bin')

# 读取大端文件
b = np.fromfile('big_endian.bin', dtype='>i4')
print('读取大端 =', b)

# 转换为本机格式
c = b.astype(np.int32)
print('转本机 =', c)

In [ ]:
# ---- np.frombuffer 字节序处理 ----
import struct

# 模拟网络数据（大端）
network_data = struct.pack('>3i', 100, 200, 300)  # 大端打包
print('原始字节 =', network_data.hex())

# 解析为大端 int32
arr = np.frombuffer(network_data, dtype='>i4')
print('大端解析 =', arr)

# 转本机
arr_native = arr.astype(np.int32)
print('本机格式 =', arr_native)

## 5. 副本和视图

In [ ]:
# ---- 无复制：赋值是引用 ----
a = np.arange(6)
print('a =', a)
print('id(a) =', id(a))

b = a  # 赋值，不是复制
print('id(b) =', id(b))  # 同一个 id

b.shape = 3, 2
print('b.shape = 3,2 后：')
print('b =')
print(b)
print('a 也变了 =')
print(a)  # a 的形状也变了

In [ ]:
# ---- 视图 view()：共享数据，但 id 不同 ----
a = np.arange(6).reshape(3, 2)
print('a =')
print(a)

b = a.view()
print('id(a) == id(b):', id(a) == id(b))  # False

# 修改 b 的形状不影响 a
b.shape = 2, 3
print('b.shape = 2,3 后：')
print('b =')
print(b)
print('a 不变 =')
print(a)

# 但修改数据会影响 a
b[0, 0] = 999
print('b[0,0]=999 后 a =')
print(a)  # a[0,0] 也变成了 999

In [ ]:
# ---- 切片也是视图 ----
a = np.arange(12)
print('a =', a)

b = a[3:]  # 切片返回视图
b[0] = 100
b[1] = 200
print('修改切片后 a =', a)  # a[3]=100, a[4]=200

In [ ]:
# ---- 副本 copy()：完全独立 ----
a = np.array([[10, 10], [2, 3], [4, 5]])
print('a =')
print(a)

b = a.copy()  # 深拷贝
print('b is a:', b is a)  # False

b[0, 0] = 100
print('b[0,0]=100 后：')
print('b =')
print(b)
print('a 不变 =')
print(a)

In [ ]:
# ---- 用 base 判断是视图还是副本 ----
a = np.arange(6)

b = a.view()       # 视图
c = a.copy()       # 副本
d = a[1:4]         # 切片（视图）

print('view  base is a:', b.base is a)   # True
print('copy  base is a:', c.base is a)   # False
print('slice base is a:', d.base is a)   # True

# base 属性指向原始数组（如果是视图）或 None（如果是独立副本）
print('view  base =', type(b.base))   # ndarray
print('copy  base =', c.base)         # None

## 6. 总结

### 字节序函数
| 函数 | 说明 |
|---|---|
| `a.byteswap(inplace=True)` | 原地交换字节序 |
| `a.byteswap(inplace=False)` | 返回交换后的新数组 |
| `a.astype('>i4')` | 转为大端 |
| `a.astype('<i4')` | 转为小端 |
| `a.dtype.newbyteorder('<')` | 创建新字节序的 dtype |
| `np.fromfile(f, dtype='>i4')` | 读取时指定字节序 |

### 副本和视图
| 操作 | 结果 | 共享数据 | 说明 |
|---|---|---|---|
| `b = a` | 引用 | 是 | 同一个对象 |
| `b = a.view()` | 视图 | 是 | id 不同，底层数据相同 |
| `b = a[i:j]` | 视图 | 是 | 切片返回视图 |
| `b = a.copy()` | 副本 | 否 | 完全独立的副本 |
| `b.base is a` | 判断 | — | True 表示 b 是 a 的视图 |